<a href="https://colab.research.google.com/github/aqrlouhanjoauhan/PakePlus-Android-v2.1.5/blob/main/youtube_subtitle_downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
#@title 🚀 YouTube Knowledge Base Cloud Extraction Engine { display-mode: "form" }
#@markdown 💡 **Smart Auto-Detection**: Enter either a **Channel URL** or a **Single Video URL** — the system will automatically handle it!

TARGET_URL = "https://www.youtube.com/@OlderBrother_" #@param {type:"string"}

from google.colab import output
js_code = """
(async () => {
    const urlParams = new URLSearchParams(window.location.search);
    return {
        target_url: urlParams.get('target_url') || urlParams.get('channel_url') || ''
    };
})();
"""
try:
    url_params = output.eval_js(js_code)
    if url_params.get('target_url'):
        TARGET_URL = url_params['target_url']
except Exception as e:
    pass

import os, re, shutil, json, subprocess, glob
print("\n⏳ Preparing cloud environment (takes ~10 seconds on first run)...")
os.system("pip install -q youtube-transcript-api yt-dlp")
from youtube_transcript_api import YouTubeTranscriptApi
from google.colab import files

base_dir = "./transcripts_temp"
if os.path.exists(base_dir):
    shutil.rmtree(base_dir)
os.makedirs(base_dir, exist_ok=True)

# Helper function: Clean text from VTT/SRT format and remove duplications
def clean_subtitle_text(vtt_text):
    lines = vtt_text.splitlines()
    cleaned = []
    for line in lines:
        line = line.strip()
        if not line or "WEBVTT" in line or "Kind:" in line or "Language:" in line or "-->" in line:
            continue
        if re.match(r'^\d+$', line):
            continue
        line = re.sub(r'<[^>]+>', '', line) # Remove inline timestamp formatting
        if not cleaned or cleaned[-1] != line:
            cleaned.append(line)
    return " ".join(cleaned)

# ==============================================================
# 3. Smart Detection: Single Video vs. Channel Playlist
# ==============================================================
is_single_video = any(kw in TARGET_URL for kw in ["watch?v=", "youtu.be/", "/shorts/"])
video_list = []
channel_title = "youtube_subtitles"

if is_single_video:
    print(f"\n🎬 Mode Detected: 【Single Video】")
    cmd = ["yt-dlp", "--dump-json", "--no-playlist", TARGET_URL]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode == 0:
        data = json.loads(res.stdout)
        video_list.append({'id': data['id'], 'title': data.get('title', data['id'])})
        safe_title = re.sub(r'[^\w\-_]', '_', data.get('title', 'video'))[:30]
        channel_title = f"Video_{safe_title}"
else:
    print(f"\n📺 Mode Detected: 【Channel / Playlist Full Batch Download】")
    fetch_url = TARGET_URL.rstrip('/')
    if not any(fetch_url.endswith(sub) for sub in ['/videos', '/shorts', '/playlists']):
        fetch_url += '/videos'

    print(f"🎯 Target URL: {fetch_url}")

    cmd = ["yt-dlp", "--flat-playlist", "--dump-single-json", fetch_url]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode == 0:
        data = json.loads(res.stdout)
        channel_title = re.sub(r'[^\w\-_]', '_', data.get('title', 'Channel'))
        for entry in data.get('entries', []):
            if entry.get('id') and entry.get('_type') != 'playlist':
                video_list.append({'id': entry['id'], 'title': entry.get('title', entry['id'])})

        total_found = len(video_list)
        print(f"✅ Successfully scanned ALL {total_found} real videos!")
        if total_found > 50:
            print(f"⚠️ Notice: >50 videos found ({total_found}). Auto-splitting into 50-video batches.")

# ==============================================================
# 4. Extract Transcripts (Robust yt-dlp Subtitle Extractor)
# ==============================================================
BATCH_SIZE = 50

if not video_list:
    print("❌ Extraction Failed: Could not recognize video info. Please check the URL.")
else:
    print("\n📝 Extracting transcripts...")
    success_count = 0
    created_zips = []

    for idx, item in enumerate(video_list, 1):
        vid = item['id']
        title = item['title']

        batch_num = ((idx - 1) // BATCH_SIZE) + 1
        batch_dir = os.path.join(base_dir, f"part_{batch_num}")
        os.makedirs(batch_dir, exist_ok=True)

        safe_title = re.sub(r'[^\w\-_]', '_', title)[:50]
        filepath = os.path.join(batch_dir, f"{idx:03d}_{safe_title}.txt")
        extracted_text = ""

        # --- Method 1: Robust yt-dlp Extractor with Broad Language Matchers ---
        try:
            v_url = f"https://www.youtube.com/watch?v={vid}"
            temp_vtt_prefix = f"/tmp/sub_{vid}"

            # Subtitle options: --write-sub (manual) + --write-auto-sub (auto), wildcard languages
            dl_cmd = [
                "yt-dlp", "--skip-download",
                "--write-sub", "--write-auto-sub",
                "--sub-langs", "en.*,zh.*,zh-Hans,zh-Hant,all",
                "--output", f"{temp_vtt_prefix}.%(ext)s", v_url
            ]
            subprocess.run(dl_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

            # Find any generated vtt/srt file
            downloaded_files = glob.glob(f"{temp_vtt_prefix}*")
            if downloaded_files:
                # Prefer English or Chinese files if multiple exist
                selected_file = downloaded_files[0]
                for df in downloaded_files:
                    if any(lang in df.lower() for lang in ['en', 'zh', 'chinese']):
                        selected_file = df
                        break

                with open(selected_file, "r", encoding="utf-8", errors="ignore") as vf:
                    raw_vtt = vf.read()
                    extracted_text = clean_subtitle_text(raw_vtt)

                # Cleanup temporary vtt files
                for df in downloaded_files:
                    try: os.remove(df)
                    except: pass
        except Exception:
            pass

        # --- Method 2: API Fallback ---
        if not extracted_text:
            try:
                t_list = YouTubeTranscriptApi.list_transcripts(vid)
                t_obj = next(iter(t_list))
                extracted_text = " ".join([e['text'] for e in t_obj.fetch()])
            except Exception:
                pass

        # --- Save Result ---
        if extracted_text.strip():
            full_content = f"Title: {title}\nURL: https://www.youtube.com/watch?v={vid}\n\n{extracted_text}"
            with open(filepath, "w", encoding="utf-8") as f:
                f.write(full_content)
            success_count += 1
            print(f"  └─ [{idx}/{len(video_list)}] ✅ Success (Part {batch_num}): {title[:25]}...")
        else:
            print(f"  └─ [{idx}/{len(video_list)}] ⚠️ Skipped (No transcript): {title[:25]}...")

    # ==============================================================
    # 5. Compress & Auto Download
    # ==============================================================
    if success_count > 0:
        print(f"\n📦 Packing finished! Successfully fetched {success_count} transcripts.")

        part_folders = [f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f))]
        part_folders.sort()

        for folder in part_folders:
            folder_path = os.path.join(base_dir, folder)
            if os.listdir(folder_path):
                if len(part_folders) == 1:
                    zip_name = f"{channel_title}_subtitles"
                else:
                    zip_name = f"{channel_title}_subtitles_{folder}"

                zip_filepath = shutil.make_archive(zip_name, 'zip', folder_path)
                created_zips.append(zip_filepath)

        print(f"🚀 Triggering download for {len(created_zips)} ZIP file(s)...")
        for zip_file in created_zips:
            print(f"  └─ Downloading: {os.path.basename(zip_file)}")
            files.download(zip_file)
    else:
        print("\n❌ Extraction Failed: Selected video(s) contain no valid transcripts.")


⏳ Preparing cloud environment (takes ~10 seconds on first run)...

📺 Mode Detected: 【Channel / Playlist Full Batch Download】
🎯 Target URL: https://www.youtube.com/@OlderBrother_/videos
✅ Successfully scanned ALL 22 real videos!

📝 Extracting transcripts...
  └─ [1/22] ✅ Success (Part 1): How To Instantly Unlock U...
  └─ [2/22] ✅ Success (Part 1): How To Stop Wasting Your ...
  └─ [3/22] ✅ Success (Part 1): why you’re smart but NOT ...
  └─ [4/22] ✅ Success (Part 1): how to win at literally A...
  └─ [5/22] ✅ Success (Part 1): how to glow up in 30 DAYS...
  └─ [6/22] ✅ Success (Part 1): how to read people ANYONE...
  └─ [7/22] ✅ Success (Part 1): how to get good at litera...
  └─ [8/22] ✅ Success (Part 1): how to become more SMART ...
  └─ [9/22] ✅ Success (Part 1): how to remember ANYTHING ...
  └─ [10/22] ✅ Success (Part 1): how to force yourself to ...
  └─ [11/22] ✅ Success (Part 1): how to learn ANYTHING fas...
  └─ [12/22] ✅ Success (Part 1): 31 tips to INSTANTLY beco...
  └─ [1

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>